In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv(r'C:\Users\AI1-PC\PycharmProjects\main\datasets\polymers\all_data.csv')

In [4]:
data_extrapol = data.dropna(axis=0)

In [16]:
smiles_array = data_extrapol['smile'].to_numpy()
gap_extrapol_array = data_extrapol['gap_extrapolated'].to_numpy()
homo_extrapol_array = data_extrapol['homo_extrapolated'].to_numpy()*-1
lumo_extrapol_array = data_extrapol['lumo_extrapolated'].to_numpy()*-1
optical_lumo_extrapol_array = data_extrapol['optical_lumo_extrapolated'].to_numpy()*-1

In [20]:
dot = 0
for s in smiles_array:
    if '.' in s:
        dot += 1
        print(s)

dot

F[C](OC(F)(F)F)C1=C2C=C(c3ccc4c(c3)C(C(F)(F)F)(C(F)(F)F)c3ccccc3-4)[SH]=C2c2sccc21.OC(F)F
CN.CN.CN1C(=O)c2cn(C)c(CNc3ccc4cc(NCc5cccc6nc7c8cccnc8c8ncccc8c7nc56)ccc4c3)c2C1=O
CN.CN.CN.CN.CNc1ccc2cc(NCc3ccc(-c4c5c(=O)n(C)c(=O)c5c(-c5ccc(CNc6ccc7cc(NCc8cccs8)ccc7c6)s5)c5c(=O)n(C)c(=O)c45)s3)ccc2c1
CN.CN.Cc1cc2csc(-c3sc(CNc4ccc5cc(NCc6sc(-c7sc(-c8scc9c8OCCO9)c8sc(C)cc78)c7c6OCCO7)ccc5c4)c4c3OCCO4)c2s1
C#Cc1sc(C#CCNc2ccc3cc(NC)ccc3c2)c2cc(C(=O)OC)c(C(=O)OC)cc12.CN.CN
CN.CN.CNc1ccc2cc(NCc3sc(-c4cc5c(cc(-c6scc7c6OCCO7)c6nsnc65)c5nsnc45)c4c3OCCO4)ccc2c1
CN.CN.CNc1ccc2cc(NCc3cc4sc(-c5sc(-c6cc7sccc7s6)c6cc7c(cc56)C(=O)C(C)(C)C7=O)cc4s3)ccc2c1
CN1C(=O)c2csc(-c3cc4c(O)c5sccc5c(O)c4s3)c2C1=O.CO.CO
CC1(C)C=C2C(=O)[Si](C)(C)C(=O)C2=C1c1ccc(CNc2ccc3cc(NCc4ccc(C5=C6C(=O)[Si](C)(C)C(=O)C6=C(c6cccs6)C5(C)C)s4)ccc3c2)s1.CN.CN
CN.CN.CNc1ccc2cc(NCc3ccc(-c4sc(-c5cccs5)c5cc6c(cc45)C(=O)N(C)C6=O)s3)ccc2c1
CN.CN.Cc1sc2c(CNc3ccc4cc(NCc5cccc6nsnc56)ccc4c3)scc2c1C
CN.CN.CNc1ccc2cc(NCc3cc4c5cc6nn(C)nc6cc5c5ccsc5c4s3

632

In [17]:
y_array = np.array([[gap_extrapol_array], [homo_extrapol_array], [lumo_extrapol_array], [optical_lumo_extrapol_array]]).T.reshape(-1, 4)

In [ ]:
import torch
from torch_geometric.data import Data
from rdkit.Chem import rdDistGeom, AllChem
from rdkit import Chem

file_number = 0
for smiles, y in zip(smiles_array, y_array):
    mol = Chem.MolFromSmiles(smiles)
    molh = Chem.AddHs(mol)

    rdDistGeom.EmbedMolecule(molh, randomSeed=42, maxAttempts=100, useRandomCoords=True)
    AllChem.MMFFOptimizeMolecule(molh, mmffVariant='MMFF94s')

    x = []
    pos = []

    for i, atom in enumerate(molh.GetAtoms()):
        x.append(atom.GetAtomicNum())
        positions = molh.GetConformer().GetAtomPosition(i)
        pos.append([positions.x, positions.y, positions.z])

    data = Data(
        x=torch.tensor(x, dtype=torch.long),
        pos=torch.tensor(pos, dtype=torch.float32),
        y=torch.tensor([[y[0], y[1], y[2]]], dtype=torch.float32)
    )

    torch.save(data, fr'C:\Users\AI1-PC\PycharmProjects\main\datasets\polymers\poly_exp_geom_to_prop_augment_test_10_v_1\{file_number}.pt')
    print(file_number)
    file_number = file_number + 1